# Higgs Boson Detection
### Kaggle Competition - Binary Classification

**Goal:** Classify events as signal (Higgs boson, label=1) or background (label=0) using 28 physics features.

**Pipeline:**
1. Data Loading & Overview
2. Exploratory Data Analysis
3. Feature Analysis
4. Model Training (XGBoost with 5-Fold CV)
5. Evaluation & Results
6. Submission

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, classification_report,
    confusion_matrix, roc_curve
)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 1. Data Loading & Overview

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Submission:  {sample_sub.shape}")
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
# Check for missing values
missing = train.isnull().sum()
print(f"Total missing values: {missing.sum()}")
if missing.sum() > 0:
    print(missing[missing > 0])

## 2. Exploratory Data Analysis

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = train["label"].value_counts()
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(["Background (0)", "Signal (1)"], counts.values, color=colors, edgecolor="black")
axes[0].set_title("Target Distribution (Count)")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(counts.values, labels=["Background (0)", "Signal (1)"],
            autopct="%1.1f%%", colors=colors, startangle=90, edgecolor="black")
axes[1].set_title("Target Distribution (%)")

plt.tight_layout()
plt.show()
print(f"Signal: {counts[1.0]} ({counts[1.0]/len(train)*100:.1f}%)")
print(f"Background: {counts[0.0]} ({counts[0.0]/len(train)*100:.1f}%)")

### 2.2 Feature Distributions

In [ ]:
feature_cols = [c for c in train.columns if c != "label"]

fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    train[train["label"] == 0][col].hist(ax=ax, bins=50, alpha=0.6, color="#e74c3c", label="Background", density=True)
    train[train["label"] == 1][col].hist(ax=ax, bins=50, alpha=0.6, color="#2ecc71", label="Signal", density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle("Feature Distributions by Class", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))
corr = train.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, annot=False, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with target
target_corr = corr["label"].drop("label").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor="black")
ax.set_xlabel("Correlation with Label")
ax.set_title("Feature Correlation with Target")
ax.axvline(x=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Top 10 features by absolute correlation with target:")
print(target_corr.head(10))

### 2.4 Box Plots by Class

In [ ]:
# Top 8 features by correlation with target
top_features = target_corr.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(top_features):
    sns.boxplot(data=train, x="label", y=col, ax=axes[i],
                palette={0.0: "#e74c3c", 1.0: "#2ecc71"})
    axes[i].set_title(col)
    axes[i].set_xticklabels(["Background", "Signal"])

plt.suptitle("Top 8 Features by Target Correlation", fontsize=16)
plt.tight_layout()
plt.show()

### 2.5 Pair Plot (Top Features)

In [ ]:
top_4 = target_corr.head(4).index.tolist()
plot_df = train[top_4 + ["label"]].copy()
plot_df["label"] = plot_df["label"].map({0.0: "Background", 1.0: "Signal"})

g = sns.pairplot(plot_df, hue="label", palette={"Background": "#e74c3c", "Signal": "#2ecc71"},
                 diag_kind="hist", plot_kws={"alpha": 0.3, "s": 10})
g.figure.suptitle("Pair Plot - Top 4 Correlated Features", y=1.02, fontsize=16)
plt.show()

## 3. Feature Analysis

In [ ]:
# Statistical tests - mean difference between classes
signal = train[train["label"] == 1]
background = train[train["label"] == 0]

stats_df = pd.DataFrame({
    "signal_mean": signal[feature_cols].mean(),
    "background_mean": background[feature_cols].mean(),
    "signal_std": signal[feature_cols].std(),
    "background_std": background[feature_cols].std(),
})
stats_df["mean_diff"] = abs(stats_df["signal_mean"] - stats_df["background_mean"])
stats_df = stats_df.sort_values("mean_diff", ascending=False)
stats_df

In [ ]:
# Highly correlated feature pairs
corr_features = corr.drop("label", axis=0).drop("label", axis=1)
upper = corr_features.where(np.triu(np.ones_like(corr_features, dtype=bool), k=1))
high_corr = [(col, row, upper.loc[row, col])
             for col in upper.columns for row in upper.index
             if abs(upper.loc[row, col]) > 0.5]
high_corr = sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)

if high_corr:
    print("Highly correlated feature pairs (|r| > 0.5):")
    for f1, f2, r in high_corr:
        print(f"  {f1} <-> {f2}: {r:.3f}")
else:
    print("No feature pairs with |correlation| > 0.5")

## 4. Model Training - XGBoost with 5-Fold CV

In [ ]:
X = train[feature_cols].values
y = train["label"].values.astype(int)
X_test = test[feature_cols].values

print(f"X_train: {X.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Labels:  {np.bincount(y)}")

In [ ]:
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

xgb_params = {
    "n_estimators": 1000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "eval_metric": "auc",
    "early_stopping_rounds": 50,
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*40} Fold {fold+1}/{n_folds} {'='*40}")
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    val_proba = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_proba
    test_preds += model.predict_proba(X_test)[:, 1] / n_folds

    auc = roc_auc_score(y_val, val_proba)
    acc = accuracy_score(y_val, (val_proba > 0.5).astype(int))
    fold_scores.append({"fold": fold+1, "auc": auc, "accuracy": acc})
    models.append(model)
    print(f"Fold {fold+1} - AUC: {auc:.5f}, Accuracy: {acc:.5f}")

print(f"\n{'='*80}")
scores_df = pd.DataFrame(fold_scores)
print(f"Mean CV AUC:      {scores_df['auc'].mean():.5f} (+/- {scores_df['auc'].std():.5f})")
print(f"Mean CV Accuracy: {scores_df['accuracy'].mean():.5f} (+/- {scores_df['accuracy'].std():.5f})")

## 5. Evaluation & Results

### 5.1 CV Scores Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC per fold
axes[0].bar(scores_df["fold"], scores_df["auc"], color="#3498db", edgecolor="black")
axes[0].axhline(y=scores_df["auc"].mean(), color="red", linestyle="--", label=f"Mean: {scores_df['auc'].mean():.4f}")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("AUC")
axes[0].set_title("AUC per Fold")
axes[0].legend()
axes[0].set_ylim(scores_df["auc"].min() - 0.005, scores_df["auc"].max() + 0.005)

# Accuracy per fold
axes[1].bar(scores_df["fold"], scores_df["accuracy"], color="#2ecc71", edgecolor="black")
axes[1].axhline(y=scores_df["accuracy"].mean(), color="red", linestyle="--", label=f"Mean: {scores_df['accuracy'].mean():.4f}")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy per Fold")
axes[1].legend()
axes[1].set_ylim(scores_df["accuracy"].min() - 0.005, scores_df["accuracy"].max() + 0.005)

plt.suptitle("Cross-Validation Results", fontsize=16)
plt.tight_layout()
plt.show()
scores_df

### 5.2 ROC Curve (OOF Predictions)

In [ ]:
fpr, tpr, thresholds = roc_curve(y, oof_preds)
oof_auc = roc_auc_score(y, oof_preds)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color="#3498db", lw=2, label=f"ROC Curve (AUC = {oof_auc:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random")
ax.fill_between(fpr, tpr, alpha=0.1, color="#3498db")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (Out-of-Fold)")
ax.legend(loc="lower right", fontsize=12)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.01])
plt.tight_layout()
plt.show()

### 5.3 Confusion Matrix

In [ ]:
oof_labels = (oof_preds > 0.5).astype(int)
cm = confusion_matrix(y, oof_labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Background", "Signal"],
            yticklabels=["Background", "Signal"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (OOF)")
plt.tight_layout()
plt.show()

print(classification_report(y, oof_labels, target_names=["Background", "Signal"]))

### 5.4 Feature Importance

In [ ]:
# Average feature importance across folds
avg_importance = np.mean([m.feature_importances_ for m in models], axis=0)
imp_df = pd.DataFrame({"feature": feature_cols, "importance": avg_importance})
imp_df = imp_df.sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(imp_df["feature"], imp_df["importance"], color="#3498db", edgecolor="black")
ax.set_xlabel("Importance (Gain)")
ax.set_title("Average Feature Importance Across 5 Folds")
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(imp_df.sort_values("importance", ascending=False).head(10).to_string(index=False))

### 5.5 Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# OOF prediction distribution
axes[0].hist(oof_preds[y == 0], bins=50, alpha=0.6, color="#e74c3c", label="Background", density=True)
axes[0].hist(oof_preds[y == 1], bins=50, alpha=0.6, color="#2ecc71", label="Signal", density=True)
axes[0].axvline(x=0.5, color="black", linestyle="--", label="Threshold (0.5)")
axes[0].set_xlabel("Predicted Probability")
axes[0].set_ylabel("Density")
axes[0].set_title("OOF Prediction Distribution")
axes[0].legend()

# Test prediction distribution
axes[1].hist(test_preds, bins=50, alpha=0.7, color="#3498db", density=True)
axes[1].axvline(x=0.5, color="black", linestyle="--", label="Threshold (0.5)")
axes[1].set_xlabel("Predicted Probability")
axes[1].set_ylabel("Density")
axes[1].set_title("Test Prediction Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Generate Submission

In [ ]:
submission = sample_sub.copy()
submission["Predicted"] = (test_preds > 0.5).astype(int)
submission.to_csv("submission.csv", index=False)

print("Submission saved to submission.csv")
print(f"\nPrediction counts:")
print(submission["Predicted"].value_counts())
print(f"\nSignal ratio: {submission['Predicted'].mean()*100:.1f}%")
submission.head(10)

## Summary

| Metric | Value |
|--------|-------|
| Model | XGBoost |
| CV Strategy | 5-Fold Stratified |
| Features | 28 (f0-f27) |
| Training Samples | 50,000 |
| Test Samples | 50,000 |